# Verified installation

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Verifies SHA256SUMS **before** installing; plans from a before-snapshot and aborts untouched on any conflict; installs both wheels `--no-deps`, then only the 8 exact pins in `requirements/kaggle-t4-core.txt` (also `--no-deps`; **no resolver ever runs**). Success: all PASS, every preinstalled package unchanged (torch, Triton, Transformers 5.x, CUDA 13.0 packages), no FlashAttention/FlashInfer, imports + `TRITON_ATTN` class resolve. Re-running changes nothing. Use `--dry-run` to see the plan only.

## 1. Setup — locate the release bundle

In [ ]:
import glob, os, pathlib
# Locate the release bundle (the Kaggle dataset holding manifests/SHA256SUMS). Override with STOKASTISK_BUNDLE.
hits = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
BUNDLE = pathlib.Path(os.environ.get("STOKASTISK_BUNDLE") or (hits[0] if hits else "")) 
assert (BUNDLE / "manifests" / "SHA256SUMS").is_file(), "attach the stokastisk-0.3.0 release dataset to this notebook"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("bundle:", BUNDLE)

## 2. Dry run (plan only; nothing is installed). **The installation cell below is blocked unless this passes.**

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE), "--dry-run"])
assert r.returncode == 0, "dry run failed: do NOT install; read the output above"
print("dry run OK")

## 3. Install

In [ ]:
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)])
RC = {"02_install": r.returncode}
print(RC)

## 4. The installed packages must be byte-identical to the bundle's wheels

In [ ]:
import importlib, importlib.metadata as md, zipfile, hashlib
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]
    mod = importlib.import_module(pkg)
    root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)} imported from {root}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}; uninstall it and re-run the install cell"

## Package the evidence

Creates `/kaggle/working/02-stokastisk-acceptance.zip` (download it from the Output tab).

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/02-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

## Verdict — this cell fails the notebook if any stage failed

In [ ]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)